In [6]:
from sqlglot.schema import MappingSchema
import json

with open("db_base_schema.json", 'r') as file:
    database_schema = MappingSchema(json.load(file))

In [33]:

with open("sql_generated_by_LLM", 'r') as file:
    llm_generation = json.load(file)
    response = json.loads(llm_generation['response'])["sql"]

    print(response)

SELECT e.FirstName, SUM(il.Quantity * il.UnitPrice) AS TotalRevenue FROM employee e JOIN customer c ON e.EmployeeId = c.SupportRepId JOIN invoiceline il ON c.CustomerId = il.InvoiceId WHERE YEAR(il.InvoiceDate) = 2025 GROUP BY e.EmployeeId ORDER BY TotalRevenue DESC LIMIT 1;


In [35]:
import json
import ollama
from sqlglot import parse_one, exp
from sqlglot.optimizer.qualify import qualify
from sqlglot.schema import MappingSchema
from sqlglot.errors import OptimizeError, ParseError

In [37]:
def validate_sql_against_schema(sql_query: str, schema: MappingSchema, dialect: str = "mysql"):
    try:
        expression = parse_one(sql_query, read=dialect)
        
        # Strictly enforce SELECT statements
        unwrapped = expression.this if isinstance(expression, exp.With) else expression
        if not isinstance(unwrapped, (exp.Select, exp.Union)):
            raise ValueError("Non-SELECT statement detected. Only read-only SELECT queries are allowed.")
        
        # Qualify columns against the schema
        qualify(expression, schema=schema)
        return True, "SQL is valid and matches schema."
        
    except ParseError as pe:
        return False, f"Syntax Error: {pe}"
    except OptimizeError as oe:
        return False, f"Schema/Validation Error: {oe}"
    except Exception as e:
        return False, f"Error: {e}"

In [38]:
is_valid, validation_message = validate_sql_against_schema(response, database_schema)

In [39]:
print(is_valid)

False


In [41]:
print(validation_message)

Schema/Validation Error: Unknown column: invoicedate


In [ ]:
def get_validated_sql(
    user_prompt: str, 
    system_prompt: str, 
    database_schema: MappingSchema, 
    llm_model: str = "llama3", # Replace with your actual model
    max_attempts: int = 3
) -> Tuple[Optional[str], str]:
    """
    Generates a PostgreSQL query using an LLM and validates it against a schema.
    If validation fails, it loops back to the LLM with error feedback for self-correction.
    
    Returns:
        (extracted_sql, status_message)
        extracted_sql will be None if all attempts fail.
    """
    current_user_prompt = user_prompt

    # 3. Begin the Self-Correction Loop
    for attempt in range(1, max_attempts + 1):
        print(f"🔄 Attempt {attempt}/{max_attempts}: Generating SQL...")
        
        try:
            # Call Ollama API
            response = ollama.generate(
                model=llm_model,
                system=system_prompt,
                prompt=current_user_prompt,
                format="json",
                options={"temperature": 0.0}  # Keep it deterministic
            )
            
            # Step A: Parse JSON Wrapper
            raw_output = response['response']
            parsed_json = json.loads(raw_output)
            extracted_sql = parsed_json.get("sql", "").strip()
            
            if not extracted_sql:
                raise ValueError("The JSON object was missing the 'sql' key or it was empty.")
            
            # Step B: Run through SQLGlot validation
            # (Inlined validation logic for self-containment)
            expression = parse_one(extracted_sql, read="postgres")
            
            # Safety check: enforce SELECT queries only
            unwrapped = expression.this if isinstance(expression, exp.With) else expression
            if not isinstance(unwrapped, (exp.Select, exp.Union)):
                raise ValueError("Security Policy Violation: Only read-only SELECT statements are allowed.")
            
            # Structural check: validate tables & columns
            qualify(expression, schema=database_schema)
            
            # If we reach this line, it passed everything!
            print(f"✅ Success on attempt {attempt}!")
            return extracted_sql, f"Successfully validated after {attempt} attempt(s)."
            
        except (json.JSONDecodeError, ParseError, OptimizeError, ValueError, Exception) as e:
            error_message = str(e)
            print(f"❌ Attempt {attempt} failed validation: {error_message}")
            
            # If we have attempts left, rewrite the prompt with explicit feedback
            if attempt < max_attempts:
                current_user_prompt = f"""### Previous Context
                {user_prompt}

                ### Instruction
                Your previous SQL generation failed validation constraints. Review the error details below, fix the issue, and regenerate the correct PostgreSQL query. 

                **Error Spotted:** {error_message}

                Provide the corrected response as the requested JSON object below:"""
            else:
                # Out of attempts
                return None, f"Failed to generate valid SQL after {max_attempts} attempts. Last error: {error_message}"